# 05 — Vector Store & RAG Pipeline Evaluation

**Phase 5 deliverable.**

Evaluates the hybrid retrieval pipeline (dense + sparse + RRF + reranker) against a
labeled test set. Runs the ablation study comparing four configurations:

| Configuration | Dense | Sparse | RRF | Reranker |
|---|---|---|---|---|
| Dense only | ✓ | | | |
| Sparse only | | ✓ | | |
| Hybrid (RRF) | ✓ | ✓ | ✓ | |
| Hybrid + reranker | ✓ | ✓ | ✓ | ✓ |

**Targets:** Recall@5 > 0.70, MRR > 0.65

In [1]:
import json
import sys
from pathlib import Path

ROOT = Path("..")
sys.path.insert(0, str(ROOT))

from src import config
from src.rag.indexer import load_chroma_collection, load_bm25_index
from src.nlp.embedder import embed_query
from src.rag.retriever import retrieve_dense, retrieve_sparse, reciprocal_rank_fusion
from src.rag.reranker import rerank

MODEL_DIR = ROOT / "models" / "sentence_transformer"

## 1. Load Indexes

In [2]:
collection = load_chroma_collection(config.CHROMADB_DIR, config.CHROMA_COLLECTION_NAME)
print(f"ChromaDB: {collection.count():,} documents")

bm25_index, bm25_chunks = load_bm25_index(config.VECTORSTORE_DIR / "bm25_index.pkl")
print(f"BM25: {len(bm25_chunks):,} chunks")

ChromaDB: 97,138 documents


BM25: 97,138 chunks


## 2. Load Retrieval Test Set

In [3]:
with open(config.EVALUATION_DIR / "retrieval_test_set.json") as f:
    test_set = json.load(f)

queries = test_set["queries"]
print(f"Loaded {len(queries)} evaluation queries")
print(f"Query types: {set(q['query_type'] for q in queries)}")
print("\nSample:")
print(json.dumps(queries[0], indent=2, ensure_ascii=False))

Loaded 94 evaluation queries
Query types: {'temporal', 'comparative', 'factual', 'thematic'}

Sample:
{
  "query": "Que visão geral este trecho oferece sobre a composição dos ativos e parte dos passivos da CSNA3?",
  "query_type": "thematic",
  "filing_id": "CSNA3_ITR_2024-06-30_1",
  "anchor_chunk_id": "CSNA3_ITR_2024-06-30_1_0058",
  "relevant_chunk_ids": [
    "CSNA3_ITR_2024-06-30_1_0058",
    "CSNA3_ITR_2024-06-30_1_0057",
    "CSNA3_ITR_2024-06-30_1_0059"
  ]
}


## 3. Retrieval Metrics — Helper Functions

In [4]:
def recall_at_k(retrieved_ids: list[str], relevant_ids: set[str], k: int) -> float:
    """Proportion of relevant docs found in the top-k retrieved."""
    if not relevant_ids:
        return 0.0
    top_k = set(retrieved_ids[:k])
    return len(top_k & relevant_ids) / len(relevant_ids)


def reciprocal_rank(retrieved_ids: list[str], relevant_ids: set[str]) -> float:
    """Reciprocal of the rank of the first relevant document."""
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_ids:
            return 1.0 / rank
    return 0.0


def ndcg_at_k(retrieved_ids: list[str], relevant_ids: set[str], k: int) -> float:
    """Normalised Discounted Cumulative Gain at k."""
    import math
    dcg = sum(
        1.0 / math.log2(rank + 2)
        for rank, doc_id in enumerate(retrieved_ids[:k])
        if doc_id in relevant_ids
    )
    ideal = sum(1.0 / math.log2(rank + 2) for rank in range(min(len(relevant_ids), k)))
    return dcg / ideal if ideal > 0 else 0.0


def evaluate_config(
    queries: list[dict],
    retrieve_fn,  # callable(query_str) -> list[RetrievedChunk]
    ks: tuple[int, ...] = (5, 10),
) -> dict:
    """Run retrieval for every query and aggregate Recall@K, MRR, NDCG@10."""
    recall_k = {k: [] for k in ks}
    mrr_scores = []
    ndcg_scores = []

    for q in queries:
        relevant = set(q["relevant_chunk_ids"])
        results = retrieve_fn(q["query"])
        ids = [c.chunk_id for c in results]

        for k in ks:
            recall_k[k].append(recall_at_k(ids, relevant, k))
        mrr_scores.append(reciprocal_rank(ids, relevant))
        ndcg_scores.append(ndcg_at_k(ids, relevant, 10))

    import numpy as np
    return {
        **{f"Recall@{k}": float(np.mean(recall_k[k])) for k in ks},
        "MRR": float(np.mean(mrr_scores)),
        "NDCG@10": float(np.mean(ndcg_scores)),
    }

## 4. Ablation Study

In [5]:
import numpy as np
import pandas as pd

# --- Configuration 1: Dense only ---
def dense_only(query: str):
    emb = embed_query(query, model_dir=MODEL_DIR)
    return retrieve_dense(emb, collection, top_k=10)

# --- Configuration 2: Sparse only ---
def sparse_only(query: str):
    return retrieve_sparse(query, bm25_index, bm25_chunks, top_k=10)

# --- Configuration 3: Hybrid RRF ---
def hybrid_rrf(query: str):
    emb = embed_query(query, model_dir=MODEL_DIR)
    dense = retrieve_dense(emb, collection, top_k=20)
    sparse = retrieve_sparse(query, bm25_index, bm25_chunks, top_k=20)
    return reciprocal_rank_fusion(dense, sparse, k=60, top_n=10)

# --- Configuration 4: Hybrid + Reranker ---
def hybrid_reranker(query: str):
    emb = embed_query(query, model_dir=MODEL_DIR)
    dense = retrieve_dense(emb, collection, top_k=20)
    sparse = retrieve_sparse(query, bm25_index, bm25_chunks, top_k=20)
    fused = reciprocal_rank_fusion(dense, sparse, k=60, top_n=10)
    return rerank(query, fused, top_k=5)

configs = [
    ("Dense only (fine-tuned BERTimbau)", dense_only),
    ("Sparse only (BM25)",               sparse_only),
    ("Hybrid (RRF)",                     hybrid_rrf),
    ("Hybrid + Reranker",                hybrid_reranker),
]

results = {}
for name, fn in configs:
    print(f"Evaluating: {name} ...")
    results[name] = evaluate_config(queries, fn)
    print(f"  {results[name]}")

df = pd.DataFrame(results).T
df = df[["Recall@5", "Recall@10", "MRR", "NDCG@10"]]
df = df.round(4)
print("\n=== Ablation Results ===")
print(df.to_string())

Evaluating: Dense only (fine-tuned BERTimbau) ...


/home/conderafael/Programing/Portfolio/cvm-intelligence/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11230.11it/s]

  {'Recall@5': 0.056737588652482254, 'Recall@10': 0.07092198581560283, 'MRR': 0.09974670719351571, 'NDCG@10': 0.06308608029183486}
Evaluating: Sparse only (BM25) ...


...

  {'Recall@5': 0.1099290780141844, 'Recall@10': 0.1453900709219858, 'MRR': 0.1727963525835866, 'NDCG@10': 0.11779470421127163}
Evaluating: Hybrid + Reranker ...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 12700.85it/s]


BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  {'Recall@5': 0.08865248226950354, 'Recall@10': 0.08865248226950354, 'MRR': 0.14804964539007093, 'NDCG@10': 0.08478091089974839}

=== Ablation Results ===
                                   Recall@5  Recall@10     MRR  NDCG@10
Dense only (fine-tuned BERTimbau)    0.0567     0.0709  0.0997   0.0631
Sparse only (BM25)                   0.1099     0.1560  0.1907   0.1252
Hybrid (RRF)                         0.1099     0.1454  0.1728   0.1178
Hybrid + Reranker                    0.0887     0.0887  0.1480   0.0848


## 5. Results Table

In [6]:
# Styled table for reporting
TARGET_RECALL5 = 0.70
TARGET_MRR = 0.65

print(df.to_markdown())

best = df.loc["Hybrid + Reranker"]
print(f"\nRecall@5 target (>0.70): {best['Recall@5']:.3f} — {'PASS' if best['Recall@5'] >= TARGET_RECALL5 else 'FAIL'}")
print(f"MRR target    (>0.65): {best['MRR']:.3f} — {'PASS' if best['MRR'] >= TARGET_MRR else 'FAIL'}")

|                                   |   Recall@5 |   Recall@10 |    MRR |   NDCG@10 |
|:----------------------------------|-----------:|------------:|-------:|----------:|
| Dense only (fine-tuned BERTimbau) |     0.0567 |      0.0709 | 0.0997 |    0.0631 |
| Sparse only (BM25)                |     0.1099 |      0.156  | 0.1907 |    0.1252 |
| Hybrid (RRF)                      |     0.1099 |      0.1454 | 0.1728 |    0.1178 |
| Hybrid + Reranker                 |     0.0887 |      0.0887 | 0.148  |    0.0848 |

Recall@5 target (>0.70): 0.089 — FAIL
MRR target    (>0.65): 0.148 — FAIL


## 6. End-to-End RAG — Sample Answers

In [7]:
from src.rag.generator import generate_answer

sample_queries = [
    "Qual foi a estratégia de crescimento da Petrobras em 2023?",
    "Como a Vale descreveu os riscos ambientais em seus relatórios?",
    "Qual foi o impacto da taxa de juros no desempenho do Itaú Unibanco?",
]

for query in sample_queries:
    print(f"\nQ: {query}")
    print("-" * 60)
    chunks = hybrid_reranker(query)
    answer = generate_answer(query, chunks)
    print(answer)
    print()


Q: Qual foi a estratégia de crescimento da Petrobras em 2023?
------------------------------------------------------------


A estratégia de crescimento da Petrobras em 2023 foi focada em "aumentar os investimentos para entregar crescimento rentável" [1], [2]. Essa abordagem foi acompanhada por:

*   **Mudanças na estratégia comercial**: Com o objetivo de aumentar a competitividade da Petrobras, trazer mais flexibilidade ao processo decisório e mais estabilidade para os consumidores [1], [2].
*   **Aperfeiçoamento da política de dividendos**: Para considerar maiores investimentos e a necessidade de manter a saúde financeira da empresa [1], [2].
*   **Aumento de investimentos (Capex)**: Em 2023, os investimentos totais aumentaram 28,7% em relação a 2022, somando US$ 12.673 milhões. Houve um aumento significativo em "Exploração & Produção" (47,9%) e "Refino, Transporte e Comercialização" (30,6%) [3], [4].


Q: Como a Vale descreveu os riscos ambientais em seus relatórios?
------------------------------------------------------------


A Vale descreve os riscos ambientais em seus relatórios da seguinte forma:

1.  **Riscos de Transição e Físicos relacionados às Mudanças Climáticas:** A empresa analisa a resiliência de seu portfólio diante dos cenários de mudanças climáticas, considerando os potenciais impactos financeiros da transição para uma economia de baixo carbono. Os riscos de transição podem resultar em impactos materiais no resultado e nos saldos contábeis, incluindo reduções de demanda de commodities devido a mudanças em políticas, ambiente regulatório (com precificação de carbono), alterações legais, tecnológicas, de mercado ou reputacionais. Também são mencionados os impactos dos riscos físicos relacionados às mudanças climáticas nos valores contábeis dos ativos [1].

2.  **Consequências de Rompimentos de Barragens:**
    *   **Fundão (Samarco):** A Vale foi alvo de acusações de crimes ambientais relacionadas a uma suposta omissão no fornecimento de informações relevantes de interesse ambiental após o romp

Informação não disponível nos documentos fornecidos.

